In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import random
import os
import urllib.request
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader

In [2]:
# --- Reproducibility --------------------------------------------------------
torch.manual_seed(42)
random.seed(42)
np.random.seed(42)

In [3]:
# --- Toy Corpus -------------------------------------------------------------
TOY_CORPUS = (
    "hello world hello help helpful hello hell helmet hello "
    "help me please hello world help world hello help hello "
    "the quick brown fox jumps over the lazy dog "
    "hello hello help helpful helpless hello world "
    "good morning good afternoon good evening good night "
    "natural language processing is fun and interesting "
    "hello hello hello help help help world world "
    "deep learning with recurrent neural networks hello "
) * 15   # repeat for sufficient training data

# --- Download Public-Domain Text --------------------------------------------
GUTENBERG_URL = "https://www.gutenberg.org/files/1342/1342-0.txt"  # Pride & Prejudice
LARGE_TEXT_FILE = "pride_prejudice.txt"

def load_large_text(path, max_chars=150000):
    """Load up to max_chars from a plain-text file, stripping Gutenberg header."""
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        raw = f.read()
    marker = "*** START OF THE PROJECT GUTENBERG EBOOK"
    idx = raw.find(marker)
    if idx != -1:
        raw = raw[idx + len(marker):]
        raw = raw[raw.find("\n")+1:]
    end_marker = "*** END OF THE PROJECT GUTENBERG EBOOK"
    end_idx = raw.find(end_marker)
    if end_idx != -1:
        raw = raw[:end_idx]
    cleaned = "".join(c for c in raw if ord(c) < 128)
    return cleaned[:max_chars]

def get_corpus(use_large=True):
    if not use_large:
        print("Using TOY corpus.")
        return TOY_CORPUS, "toy"

    if not os.path.exists(LARGE_TEXT_FILE):
        print(f"Downloading Pride & Prejudice from Project Gutenberg ...")
        try:
            urllib.request.urlretrieve(GUTENBERG_URL, LARGE_TEXT_FILE)
            print(f"Saved to {LARGE_TEXT_FILE}")
        except Exception as e:
            print(f"Download failed ({e}). Falling back to toy corpus.")
            return TOY_CORPUS, "toy"

    text = load_large_text(LARGE_TEXT_FILE)
    print(f"Large corpus loaded: {len(text):,} chars from {LARGE_TEXT_FILE}")
    return text, "large"

In [4]:
# --- Dataset ----------------------------------------------------------------
class CharDataset(Dataset):
    """
    Sliding-window character dataset.
    Each sample is (x, y) where y = x shifted right by 1 (next-char prediction).
    Teacher forcing: during training the model always receives the TRUE previous
    character as input (x), not its own prediction.
    """
    def __init__(self, text, seq_len, char2idx):
        self.seq_len = seq_len
        self.data    = [char2idx[c] for c in text if c in char2idx]

    def __len__(self):
        return max(0, len(self.data) - self.seq_len)

    def __getitem__(self, idx):
        x = torch.tensor(self.data[idx:     idx + self.seq_len],     dtype=torch.long)
        y = torch.tensor(self.data[idx + 1: idx + self.seq_len + 1], dtype=torch.long)
        return x, y

In [10]:
class CharRNN(nn.Module):
    """
    Embedding (vocab_size -> embed_dim)
    -> GRU (embed_dim -> hidden_size, num_layers)
    -> Linear (hidden_size -> vocab_size)
    Softmax is applied at generation time via temperature-scaled sampling.
    """
    def __init__(self, vocab_size, embed_dim, hidden_size,
                 num_layers=2, dropout=0.3):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers  = num_layers
        self.embedding   = nn.Embedding(vocab_size, embed_dim)
        self.gru         = nn.GRU(embed_dim, hidden_size, num_layers,
                                   batch_first=True,
                                   dropout=dropout if num_layers > 1 else 0.0)
        self.dropout_out = nn.Dropout(dropout) # Additional dropout layer
        self.fc          = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, h=None):
        emb    = self.embedding(x)        # (B, T, embed_dim)
        out, h = self.gru(emb, h)         # (B, T, hidden_size)
        out    = self.dropout_out(out)    # Apply dropout to GRU output
        logits = self.fc(out)             # (B, T, vocab_size)
        return logits, h

    def init_hidden(self, batch_size, device):
        return torch.zeros(self.num_layers, batch_size,
                           self.hidden_size, device=device)

In [6]:
# --- Training & Evaluation --------------------------------------------------
def train_epoch(model, loader, optimizer, criterion, device, clip=5.0):
    """One training epoch: teacher forcing + gradient clipping."""
    model.train()
    total_loss = 0.0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        h = model.init_hidden(x.size(0), device)
        optimizer.zero_grad()
        logits, _ = model(x, h)                               # (B, T, V)
        loss = criterion(logits.view(-1, logits.size(-1)), y.view(-1))
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), clip)    # clip gradients
        optimizer.step()
        total_loss += loss.item()
    return total_loss / max(len(loader), 1)

def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            h = model.init_hidden(x.size(0), device)
            logits, _ = model(x, h)
            loss = criterion(logits.view(-1, logits.size(-1)), y.view(-1))
            total_loss += loss.item()
    return total_loss / max(len(loader), 1)

In [7]:
# --- Temperature-Controlled Generation -------------------------------------
def generate(model, seed, char2idx, idx2char,
             length=350, temperature=1.0, device="cpu"):
    """
    Autoregressive generation with temperature scaling.
      tau < 1: sharpens distribution -> more repetitive, coherent
      tau = 1: sample from true model distribution
      tau > 1: flattens distribution -> more diverse, noisier
    """
    model.eval()
    indices = [char2idx[c] for c in seed if c in char2idx]
    if not indices:
        indices = [random.randint(0, len(char2idx) - 1)]

    x = torch.tensor([indices], dtype=torch.long, device=device)
    h = model.init_hidden(1, device)
    generated = seed

    with torch.no_grad():
        if len(indices) > 1:
            _, h = model(x[:, :-1], h)   # warm up hidden state on seed
        x = x[:, -1:]

        for _ in range(length):
            logits, h = model(x, h)                  # (1, 1, V)
            logits = logits[0, -1] / temperature      # apply temperature
            probs  = torch.softmax(logits, dim=-1).cpu().numpy().astype(np.float64)
            probs /= probs.sum()                      # renormalize for safety
            idx    = np.random.choice(len(probs), p=probs)
            generated += idx2char[idx]
            x = torch.tensor([[idx]], dtype=torch.long, device=device)

    return generated

In [8]:
# --- Plot Loss Curves -------------------------------------------------------
def plot_loss(train_losses, val_losses, tag=""):
    epochs = range(1, len(train_losses) + 1)
    plt.figure(figsize=(9, 4))
    plt.plot(epochs, train_losses, "o-",  label="Train Loss")
    plt.plot(epochs, val_losses,   "s--", label="Val Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Cross-Entropy Loss")
    plt.title(f"Character-Level GRU - Loss Curves ({tag} corpus)")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    fname = f"q1_loss_curves_{tag}.png"
    plt.savefig(fname, dpi=120)
    plt.close()
    print(f"Loss curves saved -> {fname}")
    return fname

In [12]:
# --- Main -------------------------------------------------------------------
def run(use_large=True):
    # Hyperparameters (within spec: hidden 64-256, seq 50-100, batch 64, epochs 5-20)
    SEQ_LEN     = 75
    BATCH_SIZE  = 64
    EMBED_DIM   = 64
    HIDDEN_SIZE = 256
    NUM_LAYERS  = 2
    EPOCHS      = 15
    LR          = 1e-3
    VAL_SPLIT   = 0.1
    DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    print(f"Device: {DEVICE}")

    text, tag = get_corpus(use_large=use_large)
    print(f"Corpus length: {len(text):,} characters")

    # Build vocabulary
    chars    = sorted(set(text))
    vocab    = len(chars)
    char2idx = {c: i for i, c in enumerate(chars)}
    idx2char = {i: c for c, i in char2idx.items()}
    print(f"Vocabulary size: {vocab}")

    # Train/val split
    split   = int(len(text) * (1 - VAL_SPLIT))
    tr_ds   = CharDataset(text[:split], SEQ_LEN, char2idx)
    va_ds   = CharDataset(text[split:], SEQ_LEN, char2idx)
    tr_dl   = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True,  drop_last=True)
    va_dl   = DataLoader(va_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=True)
    print(f"Train samples: {len(tr_ds):,}  |  Val samples: {len(va_ds):,}")

    # Build model
    model     = CharRNN(vocab, EMBED_DIM, HIDDEN_SIZE, NUM_LAYERS).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=LR)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)
    criterion = nn.CrossEntropyLoss()
    print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"Architecture: Embedding({vocab},{EMBED_DIM}) -> GRU(h={HIDDEN_SIZE}, "
          f"layers={NUM_LAYERS}) -> Linear -> Softmax")

    # Training loop
    tr_losses, va_losses = [], []
    print(f"\nTraining {EPOCHS} epochs ...")
    for epoch in range(1, EPOCHS + 1):
        tr = train_epoch(model, tr_dl, optimizer, criterion, DEVICE)
        va = eval_epoch(model,  va_dl, criterion, DEVICE)
        scheduler.step()
        tr_losses.append(tr)
        va_losses.append(va)
        if epoch % 3 == 0 or epoch == 1 or epoch == EPOCHS:
            print(f"  Epoch {epoch:2d}/{EPOCHS}  "
                  f"train={tr:.4f}(ppl={np.exp(tr):.1f})  "
                  f"val={va:.4f}(ppl={np.exp(va):.1f})")

    plot_loss(tr_losses, va_losses, tag)

    # Generation report
    seed = "It is a truth" if tag == "large" else "hello"
    print(f"\nTemperature-Controlled Generations  (seed: '{seed}')")
    print("=" * 60)
    for tau in [0.7, 1.0, 1.2]:
        gen = generate(model, seed, char2idx, idx2char,
                       length=350, temperature=tau, device=DEVICE)
        print(f"\n--- tau = {tau} ---")
        print(gen[:400])

    # Reflection
    print("\n" + "=" * 60)
    print("Reflection (3-5 sentences):")
    print("=" * 60)
    print("""
Sequence Length: Longer sequence lengths (75 vs 50) give the GRU more context
to learn word boundaries and sentence patterns, but consume more memory and make
gradients harder to propagate back to early positions. Too-short windows force
the model to rely on local n-gram statistics only.

Hidden Size: Increasing hidden size from 64 to 256 allows the GRU to store
richer state, reducing training loss and improving character coherence,
especially on the larger Pride & Prejudice corpus. On the small toy corpus,
256 units overfit quickly without sufficient dropout regularization.

Temperature: At tau=0.7 the softmax distribution is sharply peaked, so the model
repeatedly samples its top prediction, producing coherent but repetitive output.
At tau=1.0 we sample from the model's true learned distribution. At tau=1.2 the
distribution is flattened, causing the model to occasionally pick low-probability
characters, producing creative but sometimes grammatically incorrect text. This
demonstrates the exploration-exploitation trade-off in the sampling loop: teacher
forcing during training exposes the model only to ground-truth prefixes, so at
inference the mismatch grows with length, and temperature amplifies or suppresses
this effect.
""")


if __name__ == "__main__":
    print("\n" + "=" * 60)
    print("PHASE 1: Toy Corpus")
    print("=" * 60)
    run(use_large=False)

    print("\n" + "=" * 60)
    print("PHASE 2: Large Corpus (Pride & Prejudice ~150 KB)")
    print("=" * 60)
    run(use_large=True)



PHASE 1: Toy Corpus
Device: cuda
Using TOY corpus.
Corpus length: 5,985 characters
Vocabulary size: 27
Train samples: 5,311  |  Val samples: 524
Parameters: 650,715
Architecture: Embedding(27,64) -> GRU(h=256, layers=2) -> Linear -> Softmax

Training 15 epochs ...
  Epoch  1/15  train=1.3639(ppl=3.9)  val=0.2761(ppl=1.3)
  Epoch  3/15  train=0.0866(ppl=1.1)  val=0.0620(ppl=1.1)
  Epoch  6/15  train=0.0565(ppl=1.1)  val=0.0492(ppl=1.1)
  Epoch  9/15  train=0.0539(ppl=1.1)  val=0.0473(ppl=1.0)
  Epoch 12/15  train=0.0522(ppl=1.1)  val=0.0461(ppl=1.0)
  Epoch 15/15  train=0.0514(ppl=1.1)  val=0.0454(ppl=1.0)
Loss curves saved -> q1_loss_curves_toy.png

Temperature-Controlled Generations  (seed: 'hello')

--- tau = 0.7 ---
hello hello help helpful helpless hello world good morning good afternoon good evening good night natural language processing is fun and interesting hello hello hello help help help world world deep learning with recurrent neural networks hello hello world hello help he